# 04 -- Ablation CLAHE (spec Sec11)

Dijalankan **hanya di Base** (akurasi point tertinggi bareng Large,
99.21% -- dipilih krn lebih murah komputasi; CI 95% keempat backbone
overlap semua per notebook 03, jadi ini pilihan OPERASIONAL eksplisit,
BUKAN klaim "Base terbukti terbaik secara statistik" -- lihat CLAUDE.md).

Dua run dibandingkan, seluruh hyperparameter identik KECUALI preprocessing:
- **without_clahe**: model Base yang SUDAH ada (`fracture-runs/base_.../`,
  dari notebook 02/03) -- metriknya diambil langsung dari
  `results/metrics.json` yang sudah ter-commit, TIDAK dilatih ulang.
- **with_clahe**: run BARU -- config identik (`configs/base.yaml` +
  `base_model.yaml`) + `configs/base_clahe.yaml` (cuma menambah
  `use_clahe: true`) -- hash config otomatis beda, run_dir terpisah,
  tidak menimpa model Base yang sudah ada.

CLAHE (`clipLimit=2.0, tileGridSize=(8,8)`) diterapkan KONSISTEN di
train/val/test (bukan cuma train) via `src/fracture/data.py
make_generators(use_clahe=True)` -- lihat `src/fracture/clahe.py` utk
parameter & alasan diwarisi dari `data experiment/convnext_tiny.py`
(dead code lama, M1).

**Tidak ada ekspor ONNX di sini** -- model with_clahe murni utk
perbandingan tabel ablation, tidak dideploy ke backend produksi (yang
tetap pakai Base without_clahe yang sudah live di Cloud Run).

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Konfigurasi -- samakan dengan 02_train.ipynb/03_evaluate_export.ipynb

In [3]:
DATASET_ROOT = "/content/drive/MyDrive/project-fracture/dataset/Bone_Fracture_Dataset"
REPO_URL = "https://github.com/alianhar/project-fracture.git"
REPO_DIR = "/content/project-fracture"
RUNS_ROOT = "/content/drive/MyDrive/fracture-runs"
RESULTS_METRICS_LOCAL = f"{REPO_DIR}/results/metrics.json"  # dibaca (without_clahe) & ditulis ulang (+clahe_ablation)

N_BOOTSTRAP = 2000

In [4]:
!pip -q install pyyaml opencv-python-headless

In [5]:
import os
import shutil

def _is_valid_git_repo(path):
    return os.path.isdir(os.path.join(path, ".git"))

if os.path.exists(REPO_DIR) and not _is_valid_git_repo(REPO_DIR):
    print(f"{REPO_DIR} ada tapi bukan git repo valid -- dihapus, clone ulang.")
    shutil.rmtree(REPO_DIR)

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

assert _is_valid_git_repo(REPO_DIR), f"Clone/pull gagal -- {REPO_DIR} bukan git repo valid."

import sys
sys.path.insert(0, REPO_DIR)

Cloning into '/content/project-fracture'...
remote: Enumerating objects: 220, done.
remote: Counting objects: 100% (220/220), done.
remote: Compressing objects: 100% (201/201), done.
remote: Total 220 (delta 8), reused 187 (delta 6), pack-reused 0 (from 0)
Receiving objects: 100% (220/220), 2.60 MiB | 24.17 MiB/s, done.
Resolving deltas: 100% (8/8), done.


In [6]:
import gc
import hashlib
import json
from pathlib import Path

import numpy as np
import tensorflow as tf
import yaml

from src.fracture.data import make_generators, compute_class_weight
from src.fracture.train import run_training
from src.fracture import evaluate

## Salin dataset ke disk lokal Colab (sekali per sesi -- lewati kalau sudah pernah dari notebook lain di sesi ini)

In [7]:
import json as _json

LOCAL_DATASET_ROOT = "/content/dataset_local"
_sentinel = Path(LOCAL_DATASET_ROOT) / ".copy_done"

if not _sentinel.exists():
    with open(f"{REPO_DIR}/results/split_manifest.json") as f:
        _manifest = _json.load(f)
    n = len(_manifest["clusters"])
    print(f"Menyalin {n} gambar unik dari Drive ke disk lokal Colab...")
    for i, c in enumerate(_manifest["clusters"]):
        src = Path(DATASET_ROOT) / c["canonical_path"]
        dst = Path(LOCAL_DATASET_ROOT) / c["canonical_path"]
        dst.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(src, dst)
        if (i + 1) % 500 == 0:
            print(f"  {i + 1}/{n}")
    _sentinel.parent.mkdir(parents=True, exist_ok=True)
    _sentinel.touch()
    print("Selesai menyalin.")
else:
    print("Sudah pernah disalin di sesi ini -- lewati.")

Menyalin 3370 gambar unik dari Drive ke disk lokal Colab...
  500/3370
  1000/3370
  1500/3370
  2000/3370
  2500/3370
  3000/3370
Selesai menyalin.


## Config + training (resume-safe, sama seperti 02_train.ipynb)

In [8]:
tf.keras.utils.set_random_seed(42)

with open(f"{REPO_DIR}/configs/base.yaml") as f:
    config = yaml.safe_load(f)
with open(f"{REPO_DIR}/configs/base_model.yaml") as f:
    config.update(yaml.safe_load(f))
with open(f"{REPO_DIR}/configs/base_clahe.yaml") as f:
    config.update(yaml.safe_load(f))  # cuma menambah use_clahe: true

assert config.get("use_clahe") is True, "configs/base_clahe.yaml tidak ter-merge dgn benar"

config_str = json.dumps(config, sort_keys=True)
config_hash = hashlib.sha256(config_str.encode()).hexdigest()[:8]
run_id = f"base_clahe_{config_hash}"
run_dir = f"{RUNS_ROOT}/{run_id}"
print("run_id:", run_id)
print("run_dir:", run_dir)
print(json.dumps(config, indent=2))

run_id: base_clahe_36129f6b
run_dir: /content/drive/MyDrive/fracture-runs/base_clahe_36129f6b
{
  "seed": 42,
  "img_size": 224,
  "batch_size": 16,
  "preprocessing": "convnext_native",
  "augment_train": {
    "rotation_range": 15,
    "zoom_range": 0.15,
    "horizontal_flip": true
  },
  "split_manifest": "results/split_manifest.json",
  "use_class_weight": true,
  "phase1": {
    "epochs": 30,
    "lr": 0.0001,
    "frozen": true
  },
  "phase2": {
    "epochs": 40,
    "lr": 1e-05,
    "unfreeze_last_stages": 2
  },
  "callbacks": {
    "early_stopping": {
      "monitor": "val_loss",
      "patience": 8,
      "restore_best_weights": true
    },
    "reduce_lr": {
      "monitor": "val_loss",
      "patience": 4,
      "factor": 0.5
    },
    "checkpoint": {
      "save_best_only": true
    }
  },
  "backbone": "base",
  "use_clahe": true
}


In [9]:
manifest_path = f"{REPO_DIR}/results/split_manifest.json"

train_gen, val_gen, test_gen = make_generators(
    manifest_path=manifest_path,
    dataset_root=LOCAL_DATASET_ROOT,
    img_size=config["img_size"],
    batch_size=config["batch_size"],
    seed=config["seed"],
    augment_train=config["augment_train"],
    use_clahe=True,  # <- satu-satunya beda dari training Base yang sudah ada
)
class_weight = compute_class_weight(manifest_path, LOCAL_DATASET_ROOT) if config["use_class_weight"] else None
print("class_indices:", train_gen.class_indices)
print(f"train={train_gen.samples}  val={val_gen.samples}  test={test_gen.samples}")

Found 2358 validated image filenames belonging to 2 classes.
Found 504 validated image filenames belonging to 2 classes.
Found 508 validated image filenames belonging to 2 classes.
class_indices: {'fractured': 0, 'not_fractured': 1}
train=2358  val=504  test=508


In [10]:
best_model_path = run_training(
    backbone_name="base",
    train_gen=train_gen,
    val_gen=val_gen,
    class_weight=class_weight,
    run_dir=run_dir,
    config=config,
)
print("Model with_clahe terbaik tersimpan di:", best_model_path)

[base] Mulai dari ImageNet weights.
350926856/350926856 ━━━━━━━━━━━━━━━━━━━━ 3s 0us/step
Epoch 1/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 0s 375ms/step - accuracy: 0.6301 - loss: 0.6451

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


148/148 ━━━━━━━━━━━━━━━━━━━━ 109s 561ms/step - accuracy: 0.6993 - loss: 0.5767 - val_accuracy: 0.7956 - val_loss: 0.4984 - learning_rate: 1.0000e-04
Epoch 2/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 58s 391ms/step - accuracy: 0.8015 - loss: 0.4493 - val_accuracy: 0.8393 - val_loss: 0.4118 - learning_rate: 1.0000e-04
Epoch 3/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 65s 439ms/step - accuracy: 0.8316 - loss: 0.4108 - val_accuracy: 0.8611 - val_loss: 0.3778 - learning_rate: 1.0000e-04
Epoch 4/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 60s 406ms/step - accuracy: 0.8410 - loss: 0.3707 - val_accuracy: 0.8651 - val_loss: 0.3655 - learning_rate: 1.0000e-04
Epoch 5/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 67s 452ms/step - accuracy: 0.8605 - loss: 0.3349 - val_accuracy: 0.8849 - val_loss: 0.3343 - learning_rate: 1.0000e-04
Epoch 6/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 58s 393ms/step - accuracy: 0.8825 - loss: 0.3179 - val_accuracy: 0.8889 - val_loss: 0.3116 - learning_rate: 1.0000e-04
Epoch 7/30
148/148 ━━━━━━━━━━━━━━━━━━━━ 63s 424ms/step -

## Evaluasi with_clahe -- accuracy/f1/auroc bootstrap CI

Sama seperti notebook 03: threshold dipilih di VALIDATION (bukan test),
CI dari 2000 resample bootstrap. Cuma tiga metrik ini yang dibutuhkan
`ClaheAblationResult` (lihat `api/schemas.py`/`web/src/lib/api/types.ts`)
-- ablation tidak butuh ROC/PR/reliability/dst penuh spt notebook 03.

In [11]:
model = tf.keras.models.load_model(best_model_path)

val_gen.reset()
raw_val = model.predict(val_gen, verbose=1).squeeze(axis=-1)
prob_fractured_val = 1 - raw_val
y_fractured_val = (np.asarray(val_gen.classes) == 0).astype(int)

test_gen.reset()
raw_test = model.predict(test_gen, verbose=1).squeeze(axis=-1)
prob_fractured_test = 1 - raw_test
y_fractured_test = (np.asarray(test_gen.classes) == 0).astype(int)

threshold = evaluate.select_threshold_youden(y_fractured_val, prob_fractured_val)
ci = evaluate.bootstrap_ci(y_fractured_test, prob_fractured_test, threshold, n_resamples=N_BOOTSTRAP)

with_clahe = {
    "accuracy": {k: ci["accuracy"][k] for k in ("point", "lower", "upper")},
    "f1": {k: ci["f1"][k] for k in ("point", "lower", "upper")},
    "auroc": {k: ci["auroc"][k] for k in ("point", "lower", "upper")},
}
print("with_clahe:", json.dumps(with_clahe, indent=2))

del model
tf.keras.backend.clear_session()
gc.collect()

32/32 ━━━━━━━━━━━━━━━━━━━━ 32s 600ms/step
32/32 ━━━━━━━━━━━━━━━━━━━━ 13s 419ms/step
with_clahe: {
  "accuracy": {
    "point": 0.9901574803149606,
    "lower": 0.9803149606299213,
    "upper": 0.9980314960629921
  },
  "f1": {
    "point": 0.9882352941176471,
    "lower": 0.9767441860465116,
    "upper": 0.9976359338061466
  },
  "auroc": {
    "point": 0.9997609637939826,
    "lower": 0.999325830283531,
    "upper": 1.0
  }
}


0

## Gabungkan dgn without_clahe (dari results/metrics.json Base yang sudah ada) + simpan

In [12]:
with open(RESULTS_METRICS_LOCAL) as f:
    metrics_response = json.load(f)

base_entry = next(m for m in metrics_response["models"] if m["model_id"] == "base")
without_clahe = {
    "accuracy": {k: base_entry["accuracy"][k] for k in ("point", "lower", "upper")},
    "f1": {k: base_entry["f1"][k] for k in ("point", "lower", "upper")},
    "auroc": {k: base_entry["auroc"][k] for k in ("point", "lower", "upper")},
}

metrics_response["clahe_ablation"] = {
    "model_id": "base",
    "with_clahe": with_clahe,
    "without_clahe": without_clahe,
}

with open(RESULTS_METRICS_LOCAL, "w") as f:
    json.dump(metrics_response, f, indent=2)
print(f"Tersimpan: {RESULTS_METRICS_LOCAL}")

print("\n=== Ringkasan ablation CLAHE (Base) ===")
for label, m in [("TANPA CLAHE", without_clahe), ("DENGAN CLAHE", with_clahe)]:
    print(f"{label}: accuracy={m['accuracy']['point']:.4f} [{m['accuracy']['lower']:.4f},{m['accuracy']['upper']:.4f}]  "
          f"f1={m['f1']['point']:.4f}  auroc={m['auroc']['point']:.4f}")

def _overlap(a, b):
    return not (a["upper"] < b["lower"] or b["upper"] < a["lower"])
print("\nCI accuracy overlap (with vs without CLAHE)?", _overlap(with_clahe["accuracy"], without_clahe["accuracy"]),
      "-- overlap = TIDAK boleh klaim CLAHE signifikan berpengaruh (spec Sec7/Sec14)")

Tersimpan: /content/project-fracture/results/metrics.json

=== Ringkasan ablation CLAHE (Base) ===
TANPA CLAHE: accuracy=0.9921 [0.9843,0.9980]  f1=0.9907  auroc=0.9993
DENGAN CLAHE: accuracy=0.9902 [0.9803,0.9980]  f1=0.9882  auroc=0.9998

CI accuracy overlap (with vs without CLAHE)? True -- overlap = TIDAK boleh klaim CLAHE signifikan berpengaruh (spec Sec7/Sec14)


## Langkah selanjutnya

1. Download `results/metrics.json` yang sudah terupdate (kolom
   `clahe_ablation` sekarang terisi) dari Colab, commit ke repo -- akan
   otomatis muncul di halaman Methodology (`ClaheAblationTable`).
2. Model `with_clahe` TIDAK perlu diekspor ONNX/dideploy -- murni bukti
   ablation utk laporan skripsi.